# Insights and Policy-Relevant Reporting

This notebook generates insights and policy-relevant recommendations.

In [1]:
import pandas as pd
import numpy as np
import os

OUTPUT_FOLDER = 'output'
INPUT_FILE = os.path.join(OUTPUT_FOLDER, 'Leeds_IMD_with_Health.csv')

print("=" * 60)
print("GENERATING INSIGHTS REPORT")
print("=" * 60)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Data file not found: {INPUT_FILE}. Please run 03_Health_Outcomes.ipynb first.")

print(f"\nLoading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows")

GENERATING INSIGHTS REPORT

Loading data from output\Leeds_IMD_with_Health.csv...
Loaded 482 rows


## Summarize Deprivation Areas

In [2]:
print("\n" + "=" * 60)
print("DEPRIVATION SUMMARY")
print("=" * 60)

# Find IMD score column
imd_score_col = None
for col in df.columns:
    if 'imd' in col.lower() and 'score' in col.lower():
        imd_score_col = col
        break

# Find decile column
decile_col = None
for col in df.columns:
    if 'decile' in col.lower() and 'imd' in col.lower():
        decile_col = col
        break

# Find LSOA identifier
lsoa_id_col = None
for col in df.columns:
    if 'lsoa' in col.lower() and ('code' in col.lower() or 'name' in col.lower()):
        lsoa_id_col = col
        break

summary = {}

if imd_score_col:
    imd_scores = df[imd_score_col].dropna()
    summary['mean_imd'] = imd_scores.mean()
    summary['median_imd'] = imd_scores.median()
    summary['min_imd'] = imd_scores.min()
    summary['max_imd'] = imd_scores.max()
    
    print(f"Mean IMD Score: {summary['mean_imd']:.2f}")
    print(f"Median IMD Score: {summary['median_imd']:.2f}")
    print(f"IMD Score Range: {summary['min_imd']:.2f} - {summary['max_imd']:.2f}")
    
    # Most deprived LSOAs
    if lsoa_id_col:
        most_deprived = df.nlargest(5, imd_score_col)
        print(f"\nTop 5 Most Deprived LSOAs:")
        print(most_deprived[[lsoa_id_col, imd_score_col]])

if decile_col:
    decile_counts = df[decile_col].value_counts().sort_index()
    summary['decile_distribution'] = decile_counts.to_dict()
    
    # Count in most deprived deciles (1-3)
    most_deprived_count = df[df[decile_col].isin([1, 2, 3])].shape[0]
    total = len(df)
    summary['most_deprived_percentage'] = (most_deprived_count / total) * 100 if total > 0 else 0
    
    print(f"\nMost deprived LSOAs (Deciles 1-3): {summary['most_deprived_percentage']:.1f}%")


DEPRIVATION SUMMARY
Mean IMD Score: 27.24
Median IMD Score: 20.03
IMD Score Range: 2.02 - 78.58

Top 5 Most Deprived LSOAs:
    LSOA code (2011)  Index of Multiple Deprivation (IMD) Score
101        E01011372                                     78.577
370        E01011662                                     77.034
97         E01011368                                     75.543
94         E01011363                                     73.670
104        E01011375                                     73.056

Most deprived LSOAs (Deciles 1-3): 42.3%


## Analyze Domain Contributions

In [3]:
print("\n" + "=" * 60)
print("DOMAIN CONTRIBUTION ANALYSIS")
print("=" * 60)

# Find domain score columns
domain_names = ['Income', 'Employment', 'Education', 'Health', 'Crime', 'Housing', 'Environment']
domain_cols = {}

for domain in domain_names:
    matching_cols = [col for col in df.columns 
                    if domain.lower() in col.lower() and 'score' in col.lower()]
    if matching_cols:
        domain_cols[domain] = matching_cols[0]

if not domain_cols:
    print("Warning: Could not find domain columns")
else:
    # Calculate statistics
    domain_analysis = []
    
    for domain_name, col_name in domain_cols.items():
        domain_data = df[col_name].dropna()
        
        # Calculate coefficient of variation
        cv = (domain_data.std() / domain_data.mean()) * 100 if domain_data.mean() != 0 else 0
        
        domain_analysis.append({
            'Domain': domain_name,
            'Mean Score': domain_data.mean(),
            'Std Dev': domain_data.std(),
            'Coefficient of Variation (%)': cv,
            'Range': domain_data.max() - domain_data.min(),
            'Max Score': domain_data.max(),
            'Min Score': domain_data.min()
        })
    
    domain_df = pd.DataFrame(domain_analysis)
    domain_df = domain_df.sort_values('Mean Score', ascending=False)
    
    print("\nDomain Contribution to Deprivation:")
    print(domain_df.to_string(index=False))


DOMAIN CONTRIBUTION ANALYSIS

Domain Contribution to Deprivation:
     Domain  Mean Score   Std Dev  Coefficient of Variation (%)  Range  Max Score  Min Score
Environment   34.161012 17.044904                     49.895782 77.986     81.847      3.861
  Education   26.245494 23.619542                     89.994655 89.031     89.352      0.321
    Housing   15.231541  6.459307                     42.407440 40.992     42.332      1.340
      Crime    0.657726  0.845120                    128.491183  5.210      2.755     -2.455
     Health    0.408365  0.708222                    173.428540  4.711      3.215     -1.496
     Income    0.144060  0.110654                     76.810853  0.450      0.459      0.009
 Employment    0.111102  0.075725                     68.158167  0.336      0.343      0.007


## Analyze Deprivation-Health Patterns

In [4]:
print("\n" + "=" * 60)
print("DEPRIVATION-HEALTH PATTERN ANALYSIS")
print("=" * 60)

# Find health indicator columns
health_cols = [col for col in df.columns 
               if any(term in col.lower() for term in 
                     ['obesity', 'diabetes', 'life_expectancy', 'mental_health',
                      'smoking', 'physical_activity', 'hospital'])]

patterns = {}

if imd_score_col and health_cols:
    # Calculate correlations
    for health_col in health_cols:
        common_data = df[[imd_score_col, health_col]].dropna()
        if len(common_data) > 10:
            corr = common_data[imd_score_col].corr(common_data[health_col])
            patterns[health_col] = {
                'correlation': corr,
                'sample_size': len(common_data)
            }
    
    # Compare most vs least deprived
    if decile_col:
        most_deprived = df[df[decile_col] <= 3]  # Deciles 1-3
        least_deprived = df[df[decile_col] >= 8]  # Deciles 8-10
        
        comparison = {}
        for health_col in health_cols:
            if health_col in df.columns:
                most_mean = most_deprived[health_col].mean()
                least_mean = least_deprived[health_col].mean()
                difference = most_mean - least_mean
                pct_diff = (difference / least_mean) * 100 if least_mean != 0 else 0
                
                comparison[health_col] = {
                    'most_deprived_mean': most_mean,
                    'least_deprived_mean': least_mean,
                    'difference': difference,
                    'percent_difference': pct_diff
                }
        
        patterns['most_vs_least_deprived'] = comparison
        
        print("\nComparison between Most Deprived (Deciles 1-3) and Least Deprived (Deciles 8-10):")
        for health_indicator, comp in comparison.items():
            print(f"\n{health_indicator}:")
            print(f"  Most Deprived Mean: {comp['most_deprived_mean']:.2f}")
            print(f"  Least Deprived Mean: {comp['least_deprived_mean']:.2f}")
            print(f"  Difference: {comp['difference']:.2f} ({comp['percent_difference']:.1f}%)")


DEPRIVATION-HEALTH PATTERN ANALYSIS

Comparison between Most Deprived (Deciles 1-3) and Least Deprived (Deciles 8-10):

Obesity_Percent:
  Most Deprived Mean: 28.73
  Least Deprived Mean: 20.97
  Difference: 7.76 (37.0%)

Diabetes_Percent:
  Most Deprived Mean: 9.01
  Least Deprived Mean: 5.51
  Difference: 3.50 (63.4%)

Life_Expectancy:
  Most Deprived Mean: 79.75
  Least Deprived Mean: 81.68
  Difference: -1.94 (-2.4%)

Mental_Health_Issues_Percent:
  Most Deprived Mean: 15.93
  Least Deprived Mean: 10.75
  Difference: 5.19 (48.3%)

Smoking_Percent:
  Most Deprived Mean: 18.87
  Least Deprived Mean: 10.94
  Difference: 7.93 (72.5%)

Physical_Activity_Percent:
  Most Deprived Mean: 48.14
  Least Deprived Mean: 58.79
  Difference: -10.65 (-18.1%)

Hospital_Admissions_per_1000:
  Most Deprived Mean: 103.12
  Least Deprived Mean: 82.74
  Difference: 20.38 (24.6%)


## Generate Recommendations

In [5]:
recommendations = []

# Recommendations based on domain analysis
if 'domain_df' in locals() and len(domain_df) > 0:
    top_domain = domain_df.iloc[0]['Domain']
    recommendations.append(
        f"Priority Focus: {top_domain} deprivation is the highest contributor to overall "
        f"inequality in Leeds. Targeted interventions in this domain could have "
        f"significant impact."
    )

# Recommendations based on health patterns
if patterns and 'most_vs_least_deprived' in patterns:
    health_comparison = patterns['most_vs_least_deprived']
    
    # Find health indicators with largest differences
    largest_diffs = sorted(
        [(k, v.get('percent_difference', 0)) for k, v in health_comparison.items()],
        key=lambda x: abs(x[1]),
        reverse=True
    )
    
    if largest_diffs:
        top_health_issue = largest_diffs[0]
        recommendations.append(
            f"Health Equity Gap: The most deprived areas show {abs(top_health_issue[1]):.1f}% "
            f"difference in {top_health_issue[0]} compared to least deprived areas. "
            f"Targeted health interventions in deprived areas are urgently needed."
        )

# General recommendations
recommendations.extend([
    "Geographic Targeting: Focus resources on LSOAs in IMD deciles 1-3, "
    "which represent the most deprived areas of Leeds.",
    
    "Integrated Approach: Address multiple deprivation domains simultaneously, "
    "as they are often interconnected (e.g., income, employment, and health).",
    
    "Data-Driven Monitoring: Establish regular monitoring of health outcomes "
    "by deprivation level to track progress and identify emerging issues.",
    
    "Community Engagement: Involve residents of deprived areas in designing "
    "and implementing interventions to ensure they address local needs."
])

print("\n" + "=" * 60)
print("POLICY RECOMMENDATIONS")
print("=" * 60)
for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec}")


POLICY RECOMMENDATIONS

1. Priority Focus: Environment deprivation is the highest contributor to overall inequality in Leeds. Targeted interventions in this domain could have significant impact.

2. Health Equity Gap: The most deprived areas show 72.5% difference in Smoking_Percent compared to least deprived areas. Targeted health interventions in deprived areas are urgently needed.

3. Geographic Targeting: Focus resources on LSOAs in IMD deciles 1-3, which represent the most deprived areas of Leeds.

4. Integrated Approach: Address multiple deprivation domains simultaneously, as they are often interconnected (e.g., income, employment, and health).

5. Data-Driven Monitoring: Establish regular monitoring of health outcomes by deprivation level to track progress and identify emerging issues.

6. Community Engagement: Involve residents of deprived areas in designing and implementing interventions to ensure they address local needs.


## Generate Complete Insights Report

In [6]:
report_path = os.path.join(OUTPUT_FOLDER, 'insights_report.txt')

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("LEEDS HEALTH INEQUALITIES PROJECT - INSIGHTS REPORT\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("EXECUTIVE SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write("This report analyzes health inequalities in Leeds using the Index of ")
    f.write("Multiple Deprivation (IMD) 2019 dataset. The analysis identifies areas ")
    f.write("of highest deprivation, examines contributing factors, and explores ")
    f.write("relationships between deprivation and health outcomes.\n\n")
    
    f.write("KEY FINDINGS\n")
    f.write("-" * 80 + "\n")
    
    if summary:
        if 'most_deprived_percentage' in summary:
            f.write(f"• {summary['most_deprived_percentage']:.1f}% of LSOAs ")
            f.write(f"are in the most deprived deciles (1-3)\n")
        
        if 'mean_imd' in summary:
            f.write(f"• Mean IMD Score: {summary['mean_imd']:.2f}\n")
            f.write(f"• IMD Score Range: {summary['min_imd']:.2f} - ")
            f.write(f"{summary['max_imd']:.2f}\n")
    
    f.write("\nAREAS OF HIGHEST DEPRIVATION\n")
    f.write("-" * 80 + "\n")
    if summary and 'most_deprived_lsoas' in summary:
        f.write("Top 5 Most Deprived LSOAs:\n")
        for i, lsoa in enumerate(summary['most_deprived_lsoas'], 1):
            f.write(f"{i}. {lsoa}\n")
    
    f.write("\nDOMAIN CONTRIBUTIONS\n")
    f.write("-" * 80 + "\n")
    if 'domain_df' in locals():
        f.write("Domains ranked by mean deprivation score (higher = more deprived):\n\n")
        f.write(domain_df.to_string(index=False))
        f.write("\n\n")
    
    f.write("DEPRIVATION-HEALTH PATTERNS\n")
    f.write("-" * 80 + "\n")
    if patterns and 'most_vs_least_deprived' in patterns:
        f.write("Comparison between Most Deprived (Deciles 1-3) and ")
        f.write("Least Deprived (Deciles 8-10) Areas:\n\n")
        for health_indicator, comparison in patterns['most_vs_least_deprived'].items():
            f.write(f"{health_indicator}:\n")
            f.write(f"  Most Deprived Mean: {comparison.get('most_deprived_mean', 0):.2f}\n")
            f.write(f"  Least Deprived Mean: {comparison.get('least_deprived_mean', 0):.2f}\n")
            f.write(f"  Difference: {comparison.get('difference', 0):.2f} ")
            f.write(f"({comparison.get('percent_difference', 0):.1f}%)\n\n")
    
    f.write("POLICY RECOMMENDATIONS\n")
    f.write("-" * 80 + "\n")
    for i, rec in enumerate(recommendations, 1):
        f.write(f"{i}. {rec}\n\n")
    
    f.write("=" * 80 + "\n")
    f.write("END OF REPORT\n")
    f.write("=" * 80 + "\n")

print(f"\nInsights report saved to {report_path}")

# Summary
print("\n" + "=" * 60)
print("KEY INSIGHTS SUMMARY")
print("=" * 60)
print(f"\nTotal LSOAs analyzed: {len(df)}")

if summary and 'most_deprived_percentage' in summary:
    print(f"Most deprived LSOAs (Deciles 1-3): {summary['most_deprived_percentage']:.1f}%")

if 'domain_df' in locals() and len(domain_df) > 0:
    print(f"\nTop contributing domain: {domain_df.iloc[0]['Domain']}")

print(f"\nNumber of recommendations: {len(recommendations)}")

print("\n" + "=" * 60)
print("INSIGHTS REPORT GENERATION COMPLETE")
print("=" * 60)


Insights report saved to output\insights_report.txt

KEY INSIGHTS SUMMARY

Total LSOAs analyzed: 482
Most deprived LSOAs (Deciles 1-3): 42.3%

Top contributing domain: Environment

Number of recommendations: 6

INSIGHTS REPORT GENERATION COMPLETE
